In [26]:
# CELDA 1
import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY    = "XXXXXXXX"
STATION_ID = "IDURAN46"

hoy    = datetime.now()

url = "https://api.weather.com/v2/pws/history/all"
registros = []

for fecha in ["20261010","20261011"]:
    print(f"Descargando {fecha}...", end=" ")
    params = {
        "stationId": STATION_ID,
        "format": "json",
        "units": "m",
        "date": fecha,
        # "startDate": fecha,
        # "endDate": fecha,
        "apiKey": API_KEY,
        "numericPrecision": "decimal"
    }
    resp = requests.get(url, params=params, timeout=30)
    obs  = resp.json().get("observations", []) if resp.status_code==200 else []
    print(f"{len(obs)} registros del dia {fecha}")
    print(obs[0])
    registros.extend(obs)

Descargando 20261010... 288 registros del dia 20261010
{'stationID': 'IDURAN46', 'tz': 'America/Monterrey', 'obsTimeUtc': '2026-10-10T05:04:53Z', 'obsTimeLocal': '2026-10-10 00:04:53', 'epoch': 1791608693, 'lat': 24.035684, 'lon': -104.648617, 'solarRadiationHigh': 0.0, 'uvHigh': 0.0, 'winddirAvg': 19, 'humidityHigh': 74.0, 'humidityLow': 74.0, 'humidityAvg': 74.0, 'qcStatus': -1, 'metric': {'tempHigh': 17.8, 'tempLow': 17.8, 'tempAvg': 17.8, 'windspeedHigh': 4.2, 'windspeedLow': 3.5, 'windspeedAvg': 4.1, 'windgustHigh': 4.2, 'windgustLow': 3.5, 'windgustAvg': 4.1, 'dewptHigh': 13.1, 'dewptLow': 13.1, 'dewptAvg': 13.1, 'windchillHigh': 17.8, 'windchillLow': 17.8, 'windchillAvg': 17.8, 'heatindexHigh': 17.8, 'heatindexLow': 17.8, 'heatindexAvg': 17.8, 'pressureMax': 1010.5, 'pressureMin': 1010.5, 'pressureTrend': 0.0, 'precipRate': 0.0, 'precipTotal': 0.0}}
Descargando 20261011... 288 registros del dia 20261011
{'stationID': 'IDURAN46', 'tz': 'America/Monterrey', 'obsTimeUtc': '2026-10-

In [27]:
a = registros[0].get("metric",{}).get("tempAvg", "")
print(f"Temperatura {a}")

Temperatura 17.8


In [28]:
a = registros[0].get("humidityAvg", "")
print(f"Humedad {a}")

Humedad 74.0


In [29]:
filas = []
for o in registros:
    met = o.get("metric", {})
    filas.append({
        "created_at":  o.get("obsTimeLocal",""),
        "Temperatura": met.get("tempAvg", None),
        "Humedad":     o.get("humidityAvg", None),
    })

In [30]:
filas[0:12]

[{'created_at': '2026-10-10 00:04:53', 'Temperatura': 17.8, 'Humedad': 74.0},
 {'created_at': '2026-10-10 00:09:53', 'Temperatura': 17.8, 'Humedad': 74.0},
 {'created_at': '2026-10-10 00:14:53', 'Temperatura': 17.8, 'Humedad': 74.9},
 {'created_at': '2026-10-10 00:19:53', 'Temperatura': 17.6, 'Humedad': 75.1},
 {'created_at': '2026-10-10 00:24:53', 'Temperatura': 17.5, 'Humedad': 76.0},
 {'created_at': '2026-10-10 00:29:53', 'Temperatura': 17.5, 'Humedad': 76.0},
 {'created_at': '2026-10-10 00:34:53', 'Temperatura': 17.4, 'Humedad': 76.2},
 {'created_at': '2026-10-10 00:39:53', 'Temperatura': 17.4, 'Humedad': 77.0},
 {'created_at': '2026-10-10 00:44:53', 'Temperatura': 17.4, 'Humedad': 77.0},
 {'created_at': '2026-10-10 00:49:53', 'Temperatura': 17.3, 'Humedad': 77.4},
 {'created_at': '2026-10-10 00:54:53', 'Temperatura': 17.3, 'Humedad': 77.8},
 {'created_at': '2026-10-10 00:59:53', 'Temperatura': 17.3, 'Humedad': 77.9}]

In [31]:
df = pd.DataFrame(filas)
df["created_at"] = pd.to_datetime(df["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)
df = df.dropna(subset=["Temperatura","Humedad"])

nombre = f"IDURAN46_TH_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(nombre, index=False)
print(f"\nRegistros: {len(df)}")
print(df[["Temperatura","Humedad"]].describe().round(2))


Registros: 576
       Temperatura  Humedad
count       576.00   576.00
mean         19.68    70.54
std           3.12    13.08
min          14.40    44.90
25%          16.88    58.08
50%          19.35    73.30
75%          22.10    81.80
max          26.50    89.60
